# Convert Topologic Graph to RDF/BOT Format

This notebook demonstrates exporting topologic_fast graphs to RDF (Resource Description Framework) using the BOT (Building Topology Ontology) vocabulary.

## What is RDF/BOT?

- **RDF** (Resource Description Framework) is a standard for representing linked data
- **BOT** (Building Topology Ontology) defines concepts for building spatial structures
- Together they enable interoperability with BIM tools and semantic web applications

## BOT Core Concepts

- `bot:Site` - A geographic site containing buildings
- `bot:Building` - A building structure
- `bot:Storey` - A floor level in a building
- `bot:Space` - A 3D space (room)
- `bot:Element` - Building elements (walls, doors, windows)
- `bot:adjacentTo` - Spatial adjacency relationship

## Prerequisites

```bash
pip install rdflib plotly
```

## Note

topologic_fast does not yet have built-in RDF/BOT export. This notebook demonstrates how to manually construct RDF from graph data, which can be integrated once the feature is implemented.

## 1. Setup and Imports

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import json
import uuid

# For RDF generation
try:
    from rdflib import Graph as RDFGraph, Namespace, URIRef, Literal, BNode
    from rdflib.namespace import RDF, RDFS, XSD
    RDF_AVAILABLE = True
    print("rdflib available")
except ImportError:
    RDF_AVAILABLE = False
    print("Warning: Install rdflib for RDF export: pip install rdflib")

## 2. Define RDF Namespaces

In [ ]:
if RDF_AVAILABLE:
    # Define standard namespaces
    BOT = Namespace("https://w3id.org/bot#")
    BRICK = Namespace("https://brickschema.org/schema/Brick#")
    TOPO = Namespace("http://github.com/wassimj/topologicpy/resources#")
    
    print("Namespaces defined:")
    print(f"  BOT: {BOT}")
    print(f"  BRICK: {BRICK}")
    print(f"  TOPO: {TOPO}")
else:
    print("Skipping namespace definition - rdflib not available")

## 3. Create a Building Model

We'll create a two-storey house with multiple rooms.

In [ ]:
# Building parameters
floor_height = 3.0

# Define rooms for each floor
floor_0_rooms = [
    {"id": "lr_0", "name": "Living Room", "type": "Space", "x": 0, "y": 0, "w": 5, "l": 5},
    {"id": "kt_0", "name": "Kitchen", "type": "Space", "x": 5, "y": 0, "w": 4, "l": 5},
    {"id": "cr_0", "name": "Corridor", "type": "Space", "x": 0, "y": 5, "w": 9, "l": 2},
    {"id": "br_0", "name": "Bedroom 1", "type": "Space", "x": 0, "y": 7, "w": 4, "l": 4},
    {"id": "bt_0", "name": "Bathroom", "type": "Space", "x": 4, "y": 7, "w": 3, "l": 4},
    {"id": "ho_0", "name": "Home Office", "type": "Space", "x": 7, "y": 5, "w": 2, "l": 6},
]

floor_1_rooms = [
    {"id": "lr_1", "name": "Living Room", "type": "Space", "x": 0, "y": 0, "w": 5, "l": 5},
    {"id": "kt_1", "name": "Kitchen", "type": "Space", "x": 5, "y": 0, "w": 4, "l": 5},
    {"id": "cr_1", "name": "Corridor", "type": "Space", "x": 0, "y": 5, "w": 9, "l": 2},
    {"id": "br_1", "name": "Bedroom 2", "type": "Space", "x": 0, "y": 7, "w": 4, "l": 4},
    {"id": "bt_1", "name": "Bathroom", "type": "Space", "x": 4, "y": 7, "w": 3, "l": 4},
    {"id": "br_2", "name": "Bedroom 3", "type": "Space", "x": 7, "y": 5, "w": 2, "l": 6},
]

# Create cells for ground floor
ground_floor_cells = []
for rd in floor_0_rooms:
    cell = tf.Cell.Box(rd["x"], rd["y"], 0, rd["w"], rd["l"], floor_height)
    ground_floor_cells.append(cell)

# Create cells for first floor
first_floor_cells = []
for rd in floor_1_rooms:
    cell = tf.Cell.Box(rd["x"], rd["y"], floor_height, rd["w"], rd["l"], floor_height)
    first_floor_cells.append(cell)

# Combine into CellComplex
all_cells = ground_floor_cells + first_floor_cells
house = tf.CellComplex.ByCells(all_cells)

print(f"House created:")
print(f"  Total rooms: {house.NumCells()}")
print(f"  Total volume: {house.Volume():.1f} m³")
print(f"  Total surface area: {house.Area():.1f} m²")

# Create connectivity graph
house_graph = tf.Graph.ByTopology(house)

print(f"\nConnectivity graph:")
print(f"  Vertices: {house_graph.Order()}")
print(f"  Edges: {house_graph.Size()}")
print(f"  Density: {house_graph.Density():.3f}")

## 4. Convert Graph to JSON Structure

First, we'll create a JSON representation of the graph that captures vertices, edges, and metadata.

In [ ]:
def graph_to_json(graph, room_definitions, floor_height=3.0):
    """
    Convert a topologic_fast graph to a JSON structure.
    
    Parameters:
    - graph: tf.Graph object
    - room_definitions: List of room dicts with metadata
    - floor_height: Height of each floor
    
    Returns:
    - Dict with vertices and edges
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Map vertex positions to room definitions
    vertex_data = []
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        
        # Find matching room definition
        room_info = None
        for rd in room_definitions:
            # Calculate expected centroid
            center_x = rd["x"] + rd["w"] / 2
            center_y = rd["y"] + rd["l"] / 2
            
            if abs(coords[0] - center_x) < 0.5 and abs(coords[1] - center_y) < 0.5:
                room_info = rd
                break
        
        # Determine storey based on z coordinate
        storey = int(coords[2] / floor_height)
        
        vertex_data.append({
            "index": i,
            "id": room_info["id"] if room_info else f"v_{i}",
            "name": room_info["name"] if room_info else f"Unknown_{i}",
            "type": room_info.get("type", "Space") if room_info else "Space",
            "storey": storey,
            "coordinates": {
                "x": round(coords[0], 3),
                "y": round(coords[1], 3),
                "z": round(coords[2], 3)
            },
            "degree": graph.VertexDegree(v)
        })
    
    # Build edge data
    edge_data = []
    for i, edge in enumerate(edges):
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            # Find source and target vertex indices
            source_idx = None
            target_idx = None
            for vd in vertex_data:
                vc = vd["coordinates"]
                if abs(p1[0] - vc["x"]) < 0.01 and abs(p1[1] - vc["y"]) < 0.01 and abs(p1[2] - vc["z"]) < 0.01:
                    source_idx = vd["index"]
                if abs(p2[0] - vc["x"]) < 0.01 and abs(p2[1] - vc["y"]) < 0.01 and abs(p2[2] - vc["z"]) < 0.01:
                    target_idx = vd["index"]
            
            if source_idx is not None and target_idx is not None:
                edge_data.append({
                    "index": i,
                    "source": source_idx,
                    "target": target_idx,
                    "source_id": vertex_data[source_idx]["id"],
                    "target_id": vertex_data[target_idx]["id"]
                })
    
    return {
        "vertices": vertex_data,
        "edges": edge_data,
        "metadata": {
            "vertex_count": len(vertex_data),
            "edge_count": len(edge_data),
            "density": round(graph.Density(), 4),
            "diameter": graph.Diameter()
        }
    }

# Convert our house graph
all_room_defs = floor_0_rooms + floor_1_rooms
json_data = graph_to_json(house_graph, all_room_defs)

print("Graph JSON structure:")
print(json.dumps(json_data, indent=2)[:2000] + "...")

## 5. Convert JSON to RDF/BOT

Now we'll create RDF triples following the BOT ontology.

In [ ]:
def json_to_rdf_bot(json_data, site_name="Site_001", building_name="Building_001", namespace=None):
    """
    Convert graph JSON to RDF/BOT format.
    
    Parameters:
    - json_data: Dict from graph_to_json()
    - site_name: Name for the site
    - building_name: Name for the building
    - namespace: Base namespace URI
    
    Returns:
    - rdflib Graph with BOT triples
    """
    if not RDF_AVAILABLE:
        return None
    
    # Create RDF graph
    g = RDFGraph()
    
    # Set up namespace
    if namespace is None:
        namespace = "http://example.org/building#"
    NS = Namespace(namespace)
    
    # Bind prefixes
    g.bind("bot", BOT)
    g.bind("topo", TOPO)
    g.bind("ex", NS)
    
    # Create Site
    site_uri = NS[site_name]
    g.add((site_uri, RDF.type, BOT.Site))
    g.add((site_uri, RDFS.label, Literal(site_name)))
    
    # Create Building
    building_uri = NS[building_name]
    g.add((building_uri, RDF.type, BOT.Building))
    g.add((building_uri, RDFS.label, Literal(building_name)))
    g.add((site_uri, BOT.hasBuilding, building_uri))
    
    # Track storeys
    storey_uris = {}
    
    # Create vertices as spaces
    vertex_uris = {}
    for v in json_data["vertices"]:
        # Create or get storey
        storey_num = v["storey"]
        if storey_num not in storey_uris:
            storey_uri = NS[f"Storey_{storey_num}"]
            g.add((storey_uri, RDF.type, BOT.Storey))
            g.add((storey_uri, RDFS.label, Literal(f"Storey {storey_num}")))
            g.add((building_uri, BOT.hasStorey, storey_uri))
            storey_uris[storey_num] = storey_uri
        
        # Create space
        space_uri = NS[v["id"]]
        g.add((space_uri, RDF.type, BOT.Space))
        g.add((space_uri, RDFS.label, Literal(v["name"])))
        g.add((storey_uris[storey_num], BOT.hasSpace, space_uri))
        
        # Add coordinates as custom properties
        g.add((space_uri, TOPO.hasX, Literal(v["coordinates"]["x"], datatype=XSD.float)))
        g.add((space_uri, TOPO.hasY, Literal(v["coordinates"]["y"], datatype=XSD.float)))
        g.add((space_uri, TOPO.hasZ, Literal(v["coordinates"]["z"], datatype=XSD.float)))
        
        vertex_uris[v["index"]] = space_uri
    
    # Create edges as adjacency relationships
    for e in json_data["edges"]:
        source_uri = vertex_uris[e["source"]]
        target_uri = vertex_uris[e["target"]]
        
        # BOT adjacency (bidirectional)
        g.add((source_uri, BOT.adjacentTo, target_uri))
        g.add((target_uri, BOT.adjacentTo, source_uri))
    
    return g

if RDF_AVAILABLE:
    rdf_graph = json_to_rdf_bot(json_data, "MySite", "MyHouse", "http://example.org/house#")
    print(f"Created RDF graph with {len(rdf_graph)} triples")
else:
    print("Skipping RDF creation - rdflib not available")

## 6. Serialize RDF to Turtle Format

In [ ]:
if RDF_AVAILABLE and rdf_graph:
    # Serialize to Turtle format
    turtle_str = rdf_graph.serialize(format='turtle')
    
    print("RDF/BOT in Turtle format:")
    print("=" * 60)
    print(turtle_str)
else:
    print("RDF graph not available")

## 7. Save RDF to File

In [ ]:
if RDF_AVAILABLE and rdf_graph:
    # Save to file
    output_path = "/tmp/house_bot.ttl"
    
    rdf_graph.serialize(destination=output_path, format='turtle')
    print(f"RDF/BOT saved to: {output_path}")
    
    # Also save as JSON-LD
    jsonld_path = "/tmp/house_bot.jsonld"
    rdf_graph.serialize(destination=jsonld_path, format='json-ld')
    print(f"JSON-LD saved to: {jsonld_path}")
else:
    print("Cannot save - RDF graph not available")

## 8. Query the RDF Graph with SPARQL

In [ ]:
if RDF_AVAILABLE and rdf_graph:
    # Query: Find all spaces and their adjacencies
    query = """
    PREFIX bot: <https://w3id.org/bot#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?space ?name ?adjacent_name
    WHERE {
        ?space a bot:Space ;
               rdfs:label ?name ;
               bot:adjacentTo ?adjacent .
        ?adjacent rdfs:label ?adjacent_name .
    }
    ORDER BY ?name
    """
    
    print("SPARQL Query: Find all adjacencies")
    print("=" * 60)
    
    results = rdf_graph.query(query)
    for row in results:
        print(f"  {row.name} --> {row.adjacent_name}")
else:
    print("Cannot query - RDF graph not available")

In [ ]:
if RDF_AVAILABLE and rdf_graph:
    # Query: Find spaces on each storey
    query = """
    PREFIX bot: <https://w3id.org/bot#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?storey_name (GROUP_CONCAT(?space_name; separator=", ") AS ?spaces)
    WHERE {
        ?storey a bot:Storey ;
                rdfs:label ?storey_name ;
                bot:hasSpace ?space .
        ?space rdfs:label ?space_name .
    }
    GROUP BY ?storey_name
    ORDER BY ?storey_name
    """
    
    print("\nSPARQL Query: Spaces per storey")
    print("=" * 60)
    
    results = rdf_graph.query(query)
    for row in results:
        print(f"  {row.storey_name}: {row.spaces}")
else:
    print("Cannot query - RDF graph not available")

## 9. Visualize the Building

In [ ]:
def visualize_building_3d(cells, room_definitions, graph, floor_height=3.0):
    """
    Visualize the building with rooms and connectivity graph.
    """
    fig = go.Figure()
    
    # Color by room type
    room_colors = {
        "Living Room": "#90EE90",
        "Kitchen": "#FFDAB9",
        "Corridor": "#D3D3D3",
        "Bedroom 1": "#87CEEB",
        "Bedroom 2": "#87CEEB",
        "Bedroom 3": "#87CEEB",
        "Bathroom": "#E6E6FA",
        "Home Office": "#98FB98"
    }
    
    # Draw room boundaries (wireframe)
    for i, cell in enumerate(cells):
        rd = room_definitions[i % len(room_definitions)]
        color = room_colors.get(rd["name"], "#CCCCCC")
        
        # Get cell faces and draw edges
        faces = cell.Faces()
        for face in faces:
            verts = face.Vertices()
            coords = [v.Coordinates() for v in verts]
            
            # Draw face edges
            for j in range(len(coords)):
                p1 = coords[j]
                p2 = coords[(j + 1) % len(coords)]
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color=color, width=2),
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='red', width=4),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw graph vertices
    graph_verts = graph.Vertices()
    coords = [v.Coordinates() for v in graph_verts]
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    hover_texts = [rd["name"] for rd in room_definitions]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=8, color='red', line=dict(color='darkred', width=1)),
        name='Room Centers',
        hovertext=hover_texts,
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title='Two-Storey House with Connectivity Graph',
        scene=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700
    )
    
    return fig

fig = visualize_building_3d(all_cells, all_room_defs, house_graph)
fig.show()

## Summary

In this notebook, we demonstrated:

1. **Creating a building model** with multiple storeys using topologic_fast
2. **Converting graphs to JSON** with vertex and edge metadata
3. **Generating RDF/BOT triples** following the Building Topology Ontology
4. **Serializing to Turtle and JSON-LD** formats
5. **Querying with SPARQL** for spatial relationships
6. **Visualizing the building** in 3D

### BOT Concepts Used

- `bot:Site` - Container for the building
- `bot:Building` - The house structure
- `bot:Storey` - Floor levels (0 and 1)
- `bot:Space` - Individual rooms
- `bot:adjacentTo` - Spatial adjacency from graph edges

### Limitations

- **No built-in RDF export** - topologic_fast doesn't yet have Graph.BOTString() or Graph.ExportToBOT()
  - NOTE: These features are not yet implemented in topologic_fast
- **No geometry in RDF** - We export coordinates but not full BREP geometry
  - NOTE: BREP export is not yet implemented in topologic_fast
- **No apertures** - Doors and windows are not included in this example

### Next Steps

See RDF_BOT_Import.ipynb to learn how to import RDF/BOT back into topologic_fast graphs.